# Импорт и подготовка данных

In [148]:
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_regression, f_classif
from sklearn.preprocessing import StandardScaler

In [149]:
reg_df = pd.read_csv("../data/Regression_wine_quality_filtered.csv", sep=",", encoding="utf-8", index_col=0)
clf_df = pd.read_csv("../data/Classification_smoke_detectors_filtered.csv", sep=",", encoding="utf-8", index_col=0).head(10000)

In [150]:
X_reg = reg_df.drop('quality', axis=1)
y_reg = reg_df['quality']

X_clf = clf_df.drop('Fire Alarm', axis=1)
y_clf = clf_df['Fire Alarm']

In [151]:
scaler_reg = StandardScaler()
X_reg_scaled = scaler_reg.fit_transform(X_reg)

scaler_clf = StandardScaler()
X_clf_scaled = scaler_clf.fit_transform(X_clf)

In [152]:
X_reg_sel = SelectKBest(f_regression, k=5).fit_transform(X_reg_scaled, y_reg)

X_clf_sel = SelectKBest(f_classif, k=2).fit_transform(X_clf_scaled,y_clf)

In [153]:
kf_reg = KFold(n_splits=5, shuffle=True, random_state=42)
kf_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Sklearn MLP

In [154]:
import optuna
from hyperopt import fmin, tpe, hp, Trials, space_eval
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.neural_network import MLPClassifier, MLPRegressor
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_squared_error, r2_score

In [155]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg_sel, y_reg, test_size=0.2, random_state=42
)

In [156]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf_sel, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

## Регрессия

### Optuna

In [157]:
def optuna_obj(trial):
    hls_choice = trial.suggest_categorical('hidden_layer_sizes', ['32', '64'])
    hls = (int(hls_choice),)

    p = {
        'hidden_layer_sizes': hls,
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'solver': trial.suggest_categorical('solver', ['adam', 'sgd', 'lbfgs']),
        'alpha': trial.suggest_float('alpha', 1e-4, 1e-2, log=True),
        'max_iter': 40000,
        'random_state': 42
    }
    return cross_val_score(MLPRegressor(**p), X_reg_scaled, y_reg, cv=kf_reg, scoring='r2').mean()

reg_optuna = optuna.create_study(direction='maximize')
reg_optuna.optimize(optuna_obj, n_trials=5)
best_params = reg_optuna.best_params
best_params['hidden_layer_sizes'] = (int(reg_optuna.best_params.pop('hidden_layer_sizes')))

[I 2026-05-07 19:36:52,885] A new study created in memory with name: no-name-8721b1b9-9310-4bff-84d0-bb1aa843a54f
[I 2026-05-07 19:36:56,125] Trial 0 finished with value: 0.35025552496959367 and parameters: {'hidden_layer_sizes': '64', 'activation': 'tanh', 'solver': 'sgd', 'alpha': 0.0034585796260384644}. Best is trial 0 with value: 0.35025552496959367.
[I 2026-05-07 19:36:57,626] Trial 1 finished with value: 0.35213454687831663 and parameters: {'hidden_layer_sizes': '32', 'activation': 'tanh', 'solver': 'sgd', 'alpha': 0.00018665055346996052}. Best is trial 1 with value: 0.35213454687831663.
[I 2026-05-07 19:37:00,468] Trial 2 finished with value: 0.3419100926495432 and parameters: {'hidden_layer_sizes': '32', 'activation': 'relu', 'solver': 'adam', 'alpha': 0.0008228315079786073}. Best is trial 1 with value: 0.35213454687831663.
[I 2026-05-07 19:37:07,444] Trial 3 finished with value: 0.36469288491067064 and parameters: {'hidden_layer_sizes': '64', 'activation': 'tanh', 'solver': 'a

In [158]:
model = MLPRegressor(**best_params)
model.fit(X_train_reg, y_train_reg)

/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.",32
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'tanh'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0018932596802132406
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [159]:
y_pred = model.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred), mean_squared_error(y_test_reg, y_pred))

0.42626496046789697 0.3749392100629766


### RandomizedSearchCV

In [160]:
reg_rs = RandomizedSearchCV(
    MLPRegressor(max_iter=40000, random_state=42),
    {
        'hidden_layer_sizes': [(32,), (64,)],
        'activation': ['relu', 'tanh'],
        'solver': ['adam', 'sgd', 'lbfgs'],
        'alpha': np.logspace(-4, -2, 3)
    },
    n_iter=3, cv=kf_reg, scoring='r2', random_state=42, n_jobs=1
)

In [161]:
reg_rs.fit(X_train_reg, y_train_reg)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",MLPRegressor(...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'activation': ['relu', 'tanh'], 'alpha': array([0.0001...001 , 0.01 ]), 'hidden_layer_sizes': [(32,), (64,)], 'solver': ['adam', 'sgd', ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",3
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used h

In [162]:
y_pred = reg_rs.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred), mean_squared_error(y_test_reg, y_pred))

0.41379023163741724 0.38309151844780936


### Hyperopt

In [163]:
def hyperopt_obj(p):
    p['max_iter'] = 40000
    p['random_state'] = 42
    return -cross_val_score(MLPRegressor(**p), X_reg_scaled, y_reg, cv=kf_reg, scoring='r2').mean()

hyperopt_space = {
    'hidden_layer_sizes': hp.choice('hls', [(32,), (64,)]),
    'activation': hp.choice('act', ['relu', 'tanh']),
    'solver': hp.choice('sol', ['adam', 'sgd', 'lbfgs']),
    'alpha': hp.loguniform('alpha', np.log(1e-4), np.log(1e-2))
}

reg_hyperopt = fmin(hyperopt_obj, space=hyperopt_space, algo=tpe.suggest, max_evals=5, trials=Trials())
best_params_hyperopt_reg = space_eval(hyperopt_space, reg_hyperopt)

100%|██████████| 5/5 [01:23<00:00, 16.67s/trial, best loss: -0.34317876403179814]


In [164]:
model = MLPRegressor(**best_params_hyperopt_reg)
model.fit(X_train_reg, y_train_reg)

/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(32,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0005516698316606037
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [165]:
y_pred = model.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred), mean_squared_error(y_test_reg, y_pred))

0.30602885482410846 0.4535142096096239


## Классификация

### Optuna

In [166]:
def optuna_obj(trial):
    hls_choice = trial.suggest_categorical('hidden_layer_sizes', ['32', '64'])
    hls = (int(hls_choice),)

    p = {
        'hidden_layer_sizes': hls,
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'solver': trial.suggest_categorical('solver', ['adam', 'sgd', 'lbfgs']),
        'alpha': trial.suggest_float('alpha', 1e-4, 1e-2, log=True),
        'max_iter': 40000,
        'random_state': 42
    }
    return cross_val_score(MLPClassifier(**p), X_clf_scaled, y_clf, cv=kf_clf, scoring='accuracy').mean()

clf_optuna = optuna.create_study(direction='maximize')
clf_optuna.optimize(optuna_obj, n_trials=3)
best_params_clf_optuna = clf_optuna.best_params.copy()
best_params_clf_optuna['hidden_layer_sizes'] = (int(best_params_clf_optuna['hidden_layer_sizes']),)

[I 2026-05-07 19:39:37,857] A new study created in memory with name: no-name-34b0b3db-8393-40ab-bfba-107ee05a6880
[I 2026-05-07 19:39:41,218] Trial 0 finished with value: 0.9994 and parameters: {'hidden_layer_sizes': '64', 'activation': 'relu', 'solver': 'adam', 'alpha': 0.00011049304488786033}. Best is trial 0 with value: 0.9994.
[I 2026-05-07 19:39:41,671] Trial 1 finished with value: 0.9994000000000002 and parameters: {'hidden_layer_sizes': '32', 'activation': 'tanh', 'solver': 'lbfgs', 'alpha': 0.0013544058197472607}. Best is trial 1 with value: 0.9994000000000002.
[I 2026-05-07 19:39:42,306] Trial 2 finished with value: 0.9995 and parameters: {'hidden_layer_sizes': '64', 'activation': 'tanh', 'solver': 'lbfgs', 'alpha': 0.0002259434916135384}. Best is trial 2 with value: 0.9995.


In [167]:
model = MLPClassifier(**best_params_clf_optuna)
model.fit(X_train_clf, y_train_clf)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'tanh'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'lbfgs'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0002259434916135384
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",None


In [168]:
y_pred = model.predict(X_test_clf)
print(classification_report(y_test_clf, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       636
         1.0       1.00      1.00      1.00      1364

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



### RandomizedSearchCV

In [169]:
clf_rs = RandomizedSearchCV(
    MLPClassifier(max_iter=40000, random_state=42),
    {
        'hidden_layer_sizes': [(32,), (64,)],
        'activation': ['relu', 'tanh'],
        'solver': ['adam', 'sgd', 'lbfgs'],
        'alpha': np.logspace(-4, -2, 3)
    },
    n_iter=3, cv=kf_clf, scoring='accuracy', random_state=42, n_jobs=1
)

In [170]:
clf_rs.fit(X_train_clf, y_train_clf)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",MLPClassifier...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'activation': ['relu', 'tanh'], 'alpha': array([0.0001...001 , 0.01 ]), 'hidden_layer_sizes': [(32,), (64,)], 'solver': ['adam', 'sgd', ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",3
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be 

In [171]:
y_pred = clf_rs.predict(X_test_clf)
print(classification_report(y_test_clf, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       636
         1.0       1.00      1.00      1.00      1364

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



### Hyperopt

In [172]:
def hyperopt_obj(p):
    p['max_iter'] = 40000
    p['random_state'] = 42
    return -cross_val_score(MLPClassifier(**p), X_clf_scaled, y_clf, cv=kf_clf, scoring='accuracy').mean()

hyperopt_space = {
    'hidden_layer_sizes': hp.choice('hls', [(32,), (64,)]),
    'activation': hp.choice('act', ['relu', 'tanh']),
    'solver': hp.choice('sol', ['adam', 'sgd', 'lbfgs']),
    'alpha': hp.loguniform('alpha', np.log(1e-4), np.log(1e-2))
}

best_hyperopt = fmin(hyperopt_obj, space=hyperopt_space, algo=tpe.suggest, max_evals=3, trials=Trials())
best_params_hyperopt = space_eval(hyperopt_space, best_hyperopt)

100%|██████████| 3/3 [00:10<00:00,  3.51s/trial, best loss: -0.9994000000000002]


In [173]:
model = MLPClassifier(**best_params_hyperopt)
model.fit(X_train_clf, y_train_clf)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(32,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'tanh'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'lbfgs'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.009239575861898254
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",None


In [174]:
y_pred = model.predict(X_test_clf)
print(classification_report(y_test_clf, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       636
         1.0       1.00      1.00      1.00      1364

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



# Tensorflow

In [175]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import optuna
import keras_tuner as kt
import ray
from ray import train, tune

## Регрессия

### Optuna

In [176]:
def optuna_reg(trial):
    units = trial.suggest_int('units', 64, 256, step=64)
    opt = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop'])
    batch = trial.suggest_int('batch_size', 64, 512, step=64)
    epochs = 30
    
    model = keras.Sequential([
        layers.Dense(units, activation='relu', input_shape=(X_reg_sel.shape[1],)), 
        layers.Dense(1)
    ])
    
    model.compile(optimizer=opt, loss='mse')
    
    model.fit(X_reg_sel, y_reg, epochs=epochs, batch_size=batch, 
              validation_split=0.2, verbose=0)
    
    y_pred = model.predict(X_reg_sel, verbose=0).flatten()
    ss_res = np.sum((y_reg - y_pred) ** 2)
    ss_tot = np.sum((y_reg - np.mean(y_reg)) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    return r2 

In [177]:
study = optuna.create_study(direction='maximize')
study.optimize(optuna_reg, n_trials=5, show_progress_bar=True)

[I 2026-05-07 19:39:57,714] A new study created in memory with name: no-name-2ba4a057-3587-4bbb-a761-04d7b8863c9b
  0%|          | 0/5 [00:00<?, ?it/s]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
Best trial: 0. Best value: -2.56884:  20%|██        | 1/5 [00:02<00:08,  2.16s/it]

[I 2026-05-07 19:39:59,872] Trial 0 finished with value: -2.5688424345718492 and parameters: {'units': 64, 'optimizer': 'rmsprop', 'batch_size': 192}. Best is trial 0 with value: -2.5688424345718492.


Best trial: 1. Best value: 0.339603:  40%|████      | 2/5 [00:04<00:06,  2.26s/it]

[I 2026-05-07 19:40:02,206] Trial 1 finished with value: 0.33960316760838105 and parameters: {'units': 256, 'optimizer': 'adam', 'batch_size': 192}. Best is trial 1 with value: 0.33960316760838105.


Best trial: 1. Best value: 0.339603:  60%|██████    | 3/5 [00:06<00:04,  2.01s/it]

[I 2026-05-07 19:40:03,913] Trial 2 finished with value: -28.703715525223323 and parameters: {'units': 64, 'optimizer': 'adam', 'batch_size': 512}. Best is trial 1 with value: 0.33960316760838105.


Best trial: 1. Best value: 0.339603:  80%|████████  | 4/5 [00:07<00:01,  1.80s/it]

[I 2026-05-07 19:40:05,401] Trial 3 finished with value: -15.501758207455598 and parameters: {'units': 128, 'optimizer': 'rmsprop', 'batch_size': 448}. Best is trial 1 with value: 0.33960316760838105.


Best trial: 1. Best value: 0.339603: 100%|██████████| 5/5 [00:09<00:00,  1.83s/it]

[I 2026-05-07 19:40:06,852] Trial 4 finished with value: -15.225691813524389 and parameters: {'units': 128, 'optimizer': 'rmsprop', 'batch_size': 448}. Best is trial 1 with value: 0.33960316760838105.


In [178]:
best_params = study.best_params
print("Лучшие гиперпараметры:", best_params)

Лучшие гиперпараметры: {'units': 256, 'optimizer': 'adam', 'batch_size': 192}


In [179]:
def build_and_train_final_model(params, X, y):
    model = keras.Sequential([
        layers.Dense(params['units'], activation='relu', input_shape=(X.shape[1],)),
        layers.Dense(1)
    ])
    
    model.compile(optimizer=params['optimizer'], loss='mse')
    
    model.fit(
        X, y, 
        epochs=100, 
        batch_size=params['batch_size'],
        verbose=1
    )
    return model

In [180]:
optuna_model = build_and_train_final_model(best_params, X_train_reg, y_train_reg)

Epoch 1/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 31.6845
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 29.9671 
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 28.2116 
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 26.3137 
Epoch 5/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 24.2349 
Epoch 6/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 21.9709 
Epoch 7/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 19.5525 
Epoch 8/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 17.0440 
Epoch 9/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 14.5199 
Epoch 10/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 12.0768 
Epoch 11/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.7903  
Epoch 12/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 7.7269 
Epoch 13/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5.9430 
Epoch 14/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 4.4573 
Epoch 15/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.2488 
Epoch 16

In [181]:
y_pred = optuna_model.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
0.3802921175956726


### KerasTuner

In [182]:
def kt_reg(hp):
    model = keras.Sequential([
        keras.layers.Dense(
            hp.Int('units', 64, 256, step=64), 
            activation='relu', 
            input_shape=(X_reg_sel.shape[1],)
        ), 
        layers.Dense(1)
    ])
    
    model.compile(
        optimizer=hp.Choice('optimizer', ['adam', 'sgd', 'rmsprop']), 
        loss='mse'
    )
    return model

tuner = kt.RandomSearch(
    kt_reg, 
    objective='val_loss', 
    max_trials=2, 
    directory='kt_reg', 
    project_name='reg'
)

Reloading Tuner from kt_reg/reg/tuner0.json


In [183]:
tuner.search(X_reg_sel, y_reg, epochs=1, validation_split=0.2)
best_model = tuner.get_best_models()[0]

/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [184]:
best_model.fit(X_train_reg, y_train_reg, epochs=10)

Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4418
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4441
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4435
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4422
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4429
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4457
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4428
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4469
Epoch 9/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4425
Epoch 10/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4454


In [185]:
y_pred = best_model.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
0.3828076720237732


### Ray Tune

In [186]:
def train_model_reg(config):
    X_reg_train = config["X_data"]
    y_reg_train = config["y_data"]
    
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(config["units"], activation="relu"),
        tf.keras.layers.Dense(1, activation="linear")
    ])
    
    optimizers = {
        "adam": tf.keras.optimizers.Adam, 
        "sgd": tf.keras.optimizers.SGD, 
        "rmsprop": tf.keras.optimizers.RMSprop
    }
    
    model.compile(
        optimizer=optimizers[config["optimizer"]](learning_rate=config["lr"]), 
        loss="mean_squared_error", 
        metrics=["mae"]
    )
    
    model.fit(
        X_reg_train, y_reg_train, 
        epochs=config["epochs"], 
        batch_size=config["batch_size"], 
        verbose=0,
        validation_split=0.2
    )
    
    loss, mae = model.evaluate(X_reg_train, y_reg_train, verbose=0)
    tune.report({"loss": loss, "mae": mae})

In [187]:
param_space_reg = {
    "optimizer": tune.choice(["adam", "sgd", "rmsprop"]),
    "lr": tune.loguniform(1e-4, 1e-2),
    "units": tune.choice([16, 32, 64]),
    "batch_size": tune.choice([16, 32]),
    "epochs": 20,
    "X_data": X_train_reg,
    "y_data": y_train_reg
}

In [188]:
tuner_reg = tune.Tuner(
    train_model_reg,
    param_space=param_space_reg,
    tune_config=tune.TuneConfig(num_samples=6)
)

In [189]:
best_reg = tuner_reg.fit().get_best_result(metric="loss", mode="min")
best_config = best_reg.config
print(best_config)

(train_model_reg pid=8585) 2026-05-07 19:40:20.676017: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
(train_model_reg pid=8585) 2026-05-07 19:40:20.676057: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
(train_model_reg pid=8585) 2026-05-07 19:40:20.676067: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
(train_model_reg pid=8585) 2026-05-07 19:40:20.676116: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
(train_model_reg pid=8585) 2026-05-07 19:40:20.676276: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
(train_model_reg pid=8584) 2026-05-07 19:40:20.

{'optimizer': 'rmsprop', 'lr': 0.0012981333681452143, 'units': 64, 'batch_size': 16, 'epochs': 20, 'X_data': array([[ 0.90601191,  0.20039205,  1.05008877,  0.48302886,  1.10483337],
       [-1.77549685,  0.66254621,  3.60444201, -0.40216729,  1.38643512],
       [-0.76993107,  1.02199944, -0.98731203,  0.54204194, -0.58477711],
       ...,
       [ 0.51495855, -1.08336951,  1.17172464, -0.69723268, -0.86637886],
       [-1.83136161,  0.4057939 , -0.95690307,  0.83710732,  1.38643512],
       [-1.32857872, -0.05636026, -1.07853893, -0.69723268,  2.8883111 ]]), 'y_data': 493     6
354     6
342     6
834     5
705     5
       ..
1130    6
1294    6
860     5
1459    7
1126    6
Name: quality, Length: 1279, dtype: int64}


In [190]:
final_model = tf.keras.Sequential([
    tf.keras.layers.Dense(best_config["units"], activation="relu"),
    tf.keras.layers.Dense(1, activation="linear")
])

optimizers = {
    "adam": tf.keras.optimizers.Adam, 
    "sgd": tf.keras.optimizers.SGD, 
    "rmsprop": tf.keras.optimizers.RMSprop
}

final_model.compile(
    optimizer=optimizers[best_config["optimizer"]](learning_rate=best_config["lr"]),
    loss="mean_squared_error",
    metrics=["mae"]
)

In [191]:
history = final_model.fit(
    X_train_reg, y_train_reg,
    epochs=100,
    batch_size=best_config["batch_size"],
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 25.8310 - mae: 5.0058 - val_loss: 19.7411 - val_mae: 4.3890
Epoch 2/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 14.0309 - mae: 3.6357 - val_loss: 8.3508 - val_mae: 2.7892
Epoch 3/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 4.7851 - mae: 1.9765 - val_loss: 1.7149 - val_mae: 1.1465
Epoch 4/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.8742 - mae: 0.7480 - val_loss: 0.3482 - val_mae: 0.4481
Epoch 5/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4755 - mae: 0.5325 - val_loss: 0.3245 - val_mae: 0.4249
Epoch 6/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4795 - mae: 0.5318 - val_loss: 0.3306 - val_mae: 0.4347
Epoch 7/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4821 - mae: 0.5337 - val_loss: 0.3505 - val_mae: 0.4463
Epoch 8/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4812 - mae: 0.5352 - val_loss: 0.3325 - val_mae: 0.4308
Epoch 9/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.

In [192]:
y_pred = final_model.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
0.3666064739227295


## Классификация

### Optuna

In [193]:
def optuna_clf(trial):
    units = trial.suggest_int('units', 64, 256, step=64)
    opt = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop'])
    batch = trial.suggest_int('batch_size', 64, 512, step=64)
    epochs = 30
    
    model = keras.Sequential([
        layers.Dense(units, activation='relu', input_shape=(X_clf_sel.shape[1],)), 
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    
    model.fit(X_clf_sel, y_clf, epochs=epochs, batch_size=batch, 
              validation_split=0.2, verbose=0)
    
    y_pred_prob = model.predict(X_clf_sel, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int).ravel()
    report = classification_report(y_clf.values.ravel(), y_pred, output_dict=True)
    return report['accuracy']

In [194]:
study = optuna.create_study(direction='maximize')
study.optimize(optuna_clf, n_trials=5, show_progress_bar=True)

[I 2026-05-07 19:41:02,502] A new study created in memory with name: no-name-ce15dd11-276c-47a2-a3e6-ad6a5a292b63
  0%|          | 0/5 [00:00<?, ?it/s]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
Best trial: 0. Best value: 0.9792:  20%|██        | 1/5 [00:03<00:14,  3.68s/it]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[I 2026-05-07 19:41:06,185] Trial 0 finished with value: 0.9792 and parameters: {'units': 192, 'optimizer': 'sgd', 'batch_size': 512}. Best is trial 0 with value: 0.9792.


Best trial: 1. Best value: 0.9981:  40%|████      | 2/5 [00:09<00:14,  4.91s/it]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[I 2026-05-07 19:41:11,955] Trial 1 finished with value: 0.9981 and parameters: {'units': 256, 'optimizer': 'adam', 'batch_size': 384}. Best is trial 1 with value: 0.9981.


Best trial: 2. Best value: 0.9984:  60%|██████    | 3/5 [00:15<00:10,  5.30s/it]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[I 2026-05-07 19:41:17,715] Trial 2 finished with value: 0.9984 and parameters: {'units': 192, 'optimizer': 'adam', 'batch_size': 384}. Best is trial 2 with value: 0.9984.


Best trial: 2. Best value: 0.9984:  80%|████████  | 4/5 [00:20<00:05,  5.25s/it]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[I 2026-05-07 19:41:22,878] Trial 3 finished with value: 0.9978 and parameters: {'units': 128, 'optimizer': 'adam', 'batch_size': 448}. Best is trial 2 with value: 0.9984.


Best trial: 2. Best value: 0.9984: 100%|██████████| 5/5 [00:26<00:00,  5.33s/it]

[I 2026-05-07 19:41:29,168] Trial 4 finished with value: 0.9979 and parameters: {'units': 256, 'optimizer': 'rmsprop', 'batch_size': 256}. Best is trial 2 with value: 0.9984.


In [195]:
best_params = study.best_params
print("Лучшие гиперпараметры:", best_params)

Лучшие гиперпараметры: {'units': 192, 'optimizer': 'adam', 'batch_size': 384}


In [196]:
def build_and_train_final_model(params, X, y):
    model = keras.Sequential([
        layers.Dense(params['units'], activation='relu', input_shape=(X.shape[1],)),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=params['optimizer'], loss='binary_crossentropy', metrics=['accuracy'])
    
    model.fit(
        X, y, 
        epochs=100, 
        batch_size=params['batch_size'],
        verbose=1
    )
    return model

In [197]:
optuna_model = build_and_train_final_model(best_params, X_train_clf, y_train_clf)

Epoch 1/100


/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8595 - loss: 0.5442
Epoch 2/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9442 - loss: 0.2952
Epoch 3/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9540 - loss: 0.1875
Epoch 4/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9620 - loss: 0.1335
Epoch 5/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9710 - loss: 0.1033
Epoch 6/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9783 - loss: 0.0848
Epoch 7/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9821 - loss: 0.0729
Epoch 8/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9864 - loss: 0.0645
Epoch 9/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9894 - loss: 0.0583
Epoch 10/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9915 - loss: 0.0535
Epoch 11/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9935 - loss: 0.0498
Epoch 12/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9946 - lo

In [198]:
y_pred_prob = optuna_model.predict(X_test_clf)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()
print(classification_report(y_test_clf, y_pred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       636
         1.0       1.00      1.00      1.00      1364

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



### KerasTuner

In [199]:
import os
import shutil

In [200]:
if os.path.exists('kt_clf'):
    shutil.rmtree('kt_clf')

num_classes = len(np.unique(y_clf))

def kt_clf(hp):
    model = keras.Sequential([
        keras.layers.Dense(
            hp.Int('units', 64, 256, step=64), 
            activation='relu', 
            input_shape=(X_clf_sel.shape[1],)
        ), 
        
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=hp.Choice('optimizer', ['adam', 'sgd', 'rmsprop']), 
        loss='binary_crossentropy',  #  для бинарной
        metrics=['accuracy']
    )
    return model

In [201]:
tuner = kt.RandomSearch(
    kt_clf, 
    objective='val_loss',
    max_trials=2, 
    directory='kt_clf', 
    project_name='clf'
) 

/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [202]:
tuner.search(X_clf_sel, y_clf, epochs=1, validation_split=0.2, verbose=0)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.hypermodel.build(best_hps)

best_model.fit(X_train_clf, y_train_clf, epochs=10, validation_data=(X_test_clf, y_test_clf), verbose=1)

/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10


/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9743 - loss: 0.1303 - val_accuracy: 0.9960 - val_loss: 0.0456
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9962 - loss: 0.0362 - val_accuracy: 0.9960 - val_loss: 0.0307
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9971 - loss: 0.0277 - val_accuracy: 0.9975 - val_loss: 0.0249
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9975 - loss: 0.0239 - val_accuracy: 0.9985 - val_loss: 0.0215
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9974 - loss: 0.0214 - val_accuracy: 0.9975 - val_loss: 0.0189
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9979 - loss: 0.0197 - val_accuracy: 0.9955 - val_loss: 0.0180
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9975 - loss: 0.0187 - val_accuracy: 0.9995 - val_loss: 0.0148
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9976 - loss: 0.0176 - val_accuracy: 0.9985 - val_

In [203]:
y_pred_proba = best_model.predict(X_test_clf, verbose=0)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
print(classification_report(y_test_clf, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       636
         1.0       1.00      1.00      1.00      1364

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



### Ray Tune

In [204]:
def train_model_clf(config):
    X_clf_train = config["X_data"]
    y_clf_train = config["y_data"]
    
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(config["units"], activation="relu"),
        tf.keras.layers.Dense(config["num_classes"], activation="softmax")
    ])
    
    optimizers = {
        "adam": tf.keras.optimizers.Adam, 
        "sgd": tf.keras.optimizers.SGD, 
        "rmsprop": tf.keras.optimizers.RMSprop
    }
    
    model.compile(
        optimizer=optimizers[config["optimizer"]](learning_rate=config["lr"]), 
        loss="sparse_categorical_crossentropy", 
        metrics=["accuracy"]
    )
    
    model.fit(
        X_clf_train, y_clf_train, 
        epochs=config["epochs"], 
        batch_size=config["batch_size"], 
        verbose=0,
        validation_split=0.2
    )
    
    loss, accuracy = model.evaluate(X_clf_train, y_clf_train, verbose=0)
    tune.report({"loss": loss, "accuracy": accuracy})

In [205]:
param_space_clf = {
    "optimizer": tune.choice(["adam", "sgd", "rmsprop"]),
    "lr": tune.loguniform(1e-4, 1e-2),
    "units": tune.choice([16, 32, 64]),
    "batch_size": tune.choice([16, 32]),
    "epochs": 20,
    "num_classes": len(np.unique(y_train_clf)),
    "X_data": X_train_clf,
    "y_data": y_train_clf
}

In [206]:
tuner_clf = tune.Tuner(
    train_model_clf,
    param_space=param_space_clf,
    tune_config=tune.TuneConfig(num_samples=6)
)

In [207]:
best_clf = tuner_clf.fit().get_best_result(metric="loss", mode="min")
best_config = best_clf.config
print(best_config)

(train_model_clf pid=8619) 2026-05-07 19:42:08.648344: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
(train_model_clf pid=8619) 2026-05-07 19:42:08.648369: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
(train_model_clf pid=8619) 2026-05-07 19:42:08.648384: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
(train_model_clf pid=8619) 2026-05-07 19:42:08.648400: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
(train_model_clf pid=8619) 2026-05-07 19:42:08.648411: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
(train_model_clf pid=8616) 2026-05-07 19:42:08.

{'optimizer': 'adam', 'lr': 0.0066561068443907676, 'units': 16, 'batch_size': 32, 'epochs': 20, 'num_classes': 2, 'X_data': array([[-0.69395821,  1.35498335],
       [-0.94336019,  0.36702157],
       [-0.69227461,  1.342859  ],
       ...,
       [ 1.77855756, -1.37680719],
       [-0.67444378,  1.25868133],
       [-0.66128111,  0.7636612 ]]), 'y_data': 8911    1.0
6059    1.0
8876    1.0
5122    1.0
192     0.0
       ... 
9839    1.0
7147    1.0
1025    0.0
8633    1.0
7204    1.0
Name: Fire Alarm, Length: 8000, dtype: float64}


In [208]:
final_model = tf.keras.Sequential([
    tf.keras.layers.Dense(best_config["units"], activation="relu"),
    tf.keras.layers.Dense(best_config["num_classes"], activation="softmax")
])

optimizers = {
    "adam": tf.keras.optimizers.Adam, 
    "sgd": tf.keras.optimizers.SGD, 
    "rmsprop": tf.keras.optimizers.RMSprop
}

final_model.compile(
    optimizer=optimizers[best_config["optimizer"]](learning_rate=best_config["lr"]),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [209]:
history = final_model.fit(
    X_train_clf, y_train_clf,
    epochs=30,
    batch_size=best_config["batch_size"],
    validation_split=0.2,
    verbose=1
)

Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9791 - loss: 0.0711 - val_accuracy: 0.9981 - val_loss: 0.0256
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9961 - loss: 0.0243 - val_accuracy: 0.9987 - val_loss: 0.0174
Epoch 3/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9962 - loss: 0.0202 - val_accuracy: 0.9994 - val_loss: 0.0143
Epoch 4/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9969 - loss: 0.0180 - val_accuracy: 1.0000 - val_loss: 0.0121
Epoch 5/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9972 - loss: 0.0168 - val_accuracy: 0.9981 - val_loss: 0.0112
Epoch 6/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9969 - loss: 0.0161 - val_accuracy: 1.0000 - val_loss: 0.0101
Epoch 7/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9977 - loss: 0.0144 - val_accuracy: 1.0000 - val_loss: 0.0095
Epoch 8/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9967 - loss: 0.0135 - val_accuracy: 0.

In [210]:
y_pred = final_model.predict(X_test_clf)
y_pred_classes = np.argmax(y_pred, axis=1)
print(classification_report(y_test_clf, y_pred_classes))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
              precision    recall  f1-score   support

         0.0       1.00      0.98      0.99       636
         1.0       0.99      1.00      1.00      1364

    accuracy                           0.99      2000
   macro avg       1.00      0.99      0.99      2000
weighted avg       1.00      0.99      0.99      2000

